# Soft Active-Set Recovery Demo

This notebook isolates the new soft activity scoring step. It evaluates several `tau` values on a near-corner sample, ranks the likely active constraints, and reconstructs the best candidate vertex for each setting.

In [1]:
from __future__ import annotations

import numpy as np
from pprint import pprint

from lpas.experiments.generators import tiny_known_lp
from lpas.solver.scipy_handoff import solve_with_scipy
from lpas.solver.vertex_polishing import (
    augment_primal_constraints,
    compute_soft_activity_scores,
    generate_soft_active_set_candidates,
    reconstruct_vertex_from_active_set,
)

np.set_printoptions(suppress=True, precision=6)


In [2]:
problem = tiny_known_lp()
reference = solve_with_scipy(problem)
A_aug, b_aug = augment_primal_constraints(problem)
near_corner_sample = np.array([[1.995, 1.998]], dtype=float)
taus = [1e-1, 1e-2, 1e-3]

for tau in taus:
    slacks = b_aug - A_aug @ near_corner_sample[0]
    scores = compute_soft_activity_scores(slacks, tau=tau, method='rbf')
    candidates = generate_soft_active_set_candidates(
        near_corner_sample,
        A_aug,
        b_aug,
        problem.n,
        tau=tau,
        max_candidates_per_sample=8,
        max_total_candidates=8,
    )
    best_candidate = candidates[0]
    vertex = reconstruct_vertex_from_active_set(problem, best_candidate.active_indices)
    pprint(
        {
            'tau': tau,
            'slacks': slacks.tolist(),
            'soft_scores': scores.tolist(),
            'best_active_indices': best_candidate.active_indices,
            'reconstructed_x': None if vertex is None else vertex.x.tolist(),
            'reconstructed_objective': None if vertex is None else vertex.objective,
            'reference_x': None if reference.x is None else reference.x.tolist(),
        }
    )


{'best_active_indices': (0, 1),
 'reconstructed_objective': 10.0,
 'reconstructed_x': [2.0, 2.0],
 'reference_x': [2.0, 2.0],
 'slacks': [0.006999999999999673, 0.004999999999999893, 1.002, 1.995, 1.998],
 'soft_scores': [0.9951119854158302,
                 0.9975031223974602,
                 2.492644242367309e-44,
                 1.4115961536082898e-173,
                 4.260583748100034e-174],
 'tau': 0.1}
{'best_active_indices': (0, 1),
 'reconstructed_objective': 10.0,
 'reconstructed_x': [2.0, 2.0],
 'reference_x': [2.0, 2.0],
 'slacks': [0.006999999999999673, 0.004999999999999893, 1.002, 1.995, 1.998],
 'soft_scores': [0.6126263941844441, 0.7788007830714132, 0.0, 0.0, 0.0],
 'tau': 0.01}
{'best_active_indices': (0, 1),
 'reconstructed_objective': 10.0,
 'reconstructed_x': [2.0, 2.0],
 'reference_x': [2.0, 2.0],
 'slacks': [0.006999999999999673, 0.004999999999999893, 1.002, 1.995, 1.998],
 'soft_scores': [5.242885663387455e-22, 1.3887943864978823e-11, 0.0, 0.0, 0.0],
 'tau': 0.

## Notes

- Smaller `tau` values make the score drop off more sharply as slack grows.
- The augmented system includes both original inequality constraints and nonnegativity boundaries.
- The reconstruction step only accepts vertices that satisfy the original LP within tolerance.